# Vision Logit Lens Recipe: what does each layer already see?

The logit lens decodes a transformer's residual stream at every layer through the model's own final norm and output head (nostalgebraist 2020, "interpreting GPT: the logit lens"). A vision transformer has no language-model head, so TorchLens's default lens falls back to the model's own classification head, the readout Vilas, Schaumloeffel and Roig 2023 used to watch class evidence build up layer by layer ("Analyzing Vision Transformers for Image Classification in Class Embedding Space", NeurIPS 2023, arXiv:2310.18969). For CLIP, which has no fixed classifier, the lens decodes each vision-tower block against the text features of the label prompts captured with the image, an open-vocabulary readout in the spirit of Gandelsman, Efros and Steinhardt 2024 ("Interpreting CLIP's Image Representation via Text-Based Decomposition", ICLR 2024, arXiv:2310.05916).

Both lenses are validated: the last layer's readout must reproduce the model's own logits. Downloads: a ViT-tiny classifier (about 22 MB), CLIP ViT-B/32 (about 600 MB) and one COCO image; everything runs on a laptop CPU in about a minute.

In [ ]:
# recipe-test: tool=transformers expected=True
from __future__ import annotations

import io

import requests
import torch
from PIL import Image
from transformers import AutoImageProcessor, CLIPModel, CLIPProcessor, ViTForImageClassification

import torchlens as tl
from torchlens._semantic import logit_lens

IMAGE_URL = (
    "http://images.cocodataset.org/val2017/000000039769.jpg"  # two cats on a sofa (COCO val2017)
)
image = Image.open(io.BytesIO(requests.get(IMAGE_URL, timeout=60).content)).convert("RGB")
VIT_ID, VIT_REVISION = (
    "WinKawaks/vit-tiny-patch16-224",
    "77d1485af66b34d4ed0fe95dbb0c60c7496f950b",
)  # pragma: allowlist secret
CLIP_ID, CLIP_REVISION = (
    "openai/clip-vit-base-patch32",
    "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268",
)  # pragma: allowlist secret
RESULT = True

## A ViT classifier: the class each layer already votes for

`logit_lens(trace)` finds the classification head, sends every encoder block's `[CLS]` state through the final norm and the classifier, and checks that the last block reproduces the captured logits (`is_validated`).

In [ ]:
# recipe-test: tool=transformers expected=True
processor = AutoImageProcessor.from_pretrained(VIT_ID, revision=VIT_REVISION)
vit = ViTForImageClassification.from_pretrained(
    VIT_ID, revision=VIT_REVISION, attn_implementation="eager"
).eval()
pixels = processor(images=image, return_tensors="pt")["pixel_values"]
trace = tl.trace(vit, (), input_kwargs={"pixel_values": pixels})
result = logit_lens(trace)
print("lens source:", result.lens_source, "| validated:", result.is_validated)
assert result.lens_source == "classifier_head" and result.is_validated

labels = vit.config.id2label
for address, top in result.top_tokens(3):
    print(f"{address:>24}", [(labels[index].split(",")[0], round(prob, 3)) for index, prob in top])
RESULT = result.lens_source == "classifier_head" and result.is_validated

The last row is the model's answer. The rows above show at which depth the answer first appears, and what the earlier blocks would have said instead.

In [ ]:
# recipe-test: tool=transformers expected=True
answer = int(result.entries[-1].logits.argmax())
first_layer = next(
    index
    for index, entry in enumerate(result.entries)
    if int(entry.logits.argmax()) == answer
    and all(int(e.logits.argmax()) == answer for e in result.entries[index:])
)
print(f"answer: {labels[answer]!r}, settled from block {first_layer} of {len(result.entries) - 1}")
top3_final = [labels[index].lower() for index, _ in result.top_tokens(3)[-1][1]]
assert any("cat" in label for label in top3_final), top3_final
RESULT = any("cat" in label for label in top3_final)

## CLIP: an open-vocabulary lens

Capture CLIP on the image and on the label prompts you want to score. The lens then decodes every vision-tower block against the captured text features with CLIP's own logit scale, so each layer gets a score per prompt and the last layer equals `logits_per_image`.

In [ ]:
# recipe-test: tool=transformers expected=True
clip = CLIPModel.from_pretrained(
    CLIP_ID, revision=CLIP_REVISION, attn_implementation="eager"
).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_ID, revision=CLIP_REVISION)
prompts = [
    "a photo of a cat",
    "a photo of a dog",
    "a photo of a sofa",
    "a photo of a remote control",
    "a photo of a car",
]
inputs = clip_processor(text=prompts, images=image, return_tensors="pt", padding=True)
clip_trace = tl.trace(clip, (), input_kwargs=dict(inputs))
clip_result = logit_lens(clip_trace)
print("lens source:", clip_result.lens_source, "| validated:", clip_result.is_validated)
assert clip_result.lens_source == "contrastive_head" and clip_result.is_validated

for address, top in clip_result.top_tokens(2):
    print(f"{address:>34}", [(prompts[index], round(prob, 3)) for index, prob in top])
with torch.no_grad():
    expected = clip(**inputs).logits_per_image
assert torch.allclose(clip_result.entries[-1].logits, expected, atol=1e-4)
assert prompts[int(expected.argmax())] == "a photo of a cat"
RESULT = (
    clip_result.lens_source == "contrastive_head"
    and prompts[int(expected.argmax())] == "a photo of a cat"
)